# Genome-wide QC + thinning (Google Batch / dsub)

`03a_genome_wide_qc_thinning_merge.ipynb`'s interactive concurrent driver reads
all 22 ACAF chromosomes from the same gcsfuse-mounted network source at once --
real usage (`top` on the VM while it ran) showed `%Cpu(s)` sitting near-100% idle
with every `plink2` process stalled at <1% CPU while `gcsfuse` itself was the
single busiest process on the machine: classic network-mount contention, not a
CPU problem, so bumping `N_CONCURRENT` further doesn't help. Each Batch task gets
its own dedicated VM **and its own network path** to ACAF, sidestepping that
contention entirely -- in principle wall-clock collapses toward the
single-chromosome QC time (~10-20 min) instead of hours, since chromosomes
genuinely run in parallel rather than queuing on one mount.

**One real unknown this notebook exists to resolve before spending real compute:**
`03b_grm_shard_batch_submit.ipynb`'s Batch tasks only ever read/wrote the
*workspace* bucket (this project's own bucket, which the Batch service account
already has access to). This notebook needs Batch tasks to read directly from the
**CDR's own controlled-tier ACAF bucket** instead -- a different, more restricted
resource. Whether VPC Service Controls / controlled-tier data governance actually
permits that from a Batch worker (vs. only from the interactive VM's own gcsfuse
mount) is untested. Stage 0 below checks exactly this, cheaply, before Stage 1/2
risk any real per-chromosome compute.

## Prerequisites

Same as `03b_grm_shard_batch_submit.ipynb` -- `dsub` submits from this notebook's
VM, but the actual work happens on separate Batch-provisioned machines. Needs
`dsub` installed and `gcloud` already authenticated (should already be true on a
Verily Workbench VM).

In [ ]:
%%bash
set -e

if ! command -v dsub >/dev/null 2>&1; then
  pip install --quiet dsub
fi
dsub --version

echo "--- gcloud config ---"
gcloud config list --format='text(core.project,compute.region)' 2>&1 || true

## Inputs

`PROJECT_ID`/`REGION`/`SERVICE_ACCOUNT`/`NETWORK`/`SUBNETWORK`/`WORKSPACE_BUCKET_GS`:
same values `03b_grm_shard_batch_submit.ipynb` already confirmed working for this
project -- reused as-is, not re-derived.

`ACAF_BUCKET_GS` is an **unconfirmed placeholder** -- the `gs://` form of
`CDR_ACAF_PGEN_DIR["v9"]` (`03a_genome_wide_qc_thinning_merge.ipynb`'s own
gcsfuse-mounted path). Find the real value the same way `WORKSPACE_BUCKET_GS` was
found for the workspace bucket: `ps -eo pid,args | grep gcsfuse` on this VM, and
look for the mount matching `~/workspace/cdrv9/vwb-aou-datasets-controlled-v9` --
the `gcsfuse <bucket-name> <mount-path>` invocation shows the real bucket name.
**Do not guess this value or run Stage 1+ until it's confirmed** -- an unconfirmed
bucket name is exactly the class of bug `03b_grm_shard_batch_submit.ipynb`'s own
Status notes describe hitting (a guessed bucket name that doesn't exist causes a
job to run and then fail invisibly at the log-upload step).

In [ ]:
import os

# same values 03b_grm_shard_batch_submit.ipynb already confirmed working -- reused,
# not re-derived
PROJECT_ID = "wb-swift-sprout-7231"
REGION = "us-central1"
SERVICE_ACCOUNT = "pet-27799165194323faf22e2@wb-swift-sprout-7231.iam.gserviceaccount.com"
NETWORK = f"projects/{PROJECT_ID}/global/networks/network"
SUBNETWORK = f"projects/{PROJECT_ID}/regions/{REGION}/subnetworks/subnetwork"
WORKSPACE_BUCKET_GS = "gs://cloned-shared-env-pilot-wb-swift-sprout-7231"

WORKSPACE_BUCKET = os.path.expanduser(
    "~/workspace/Data from All of Us Controlled Tier /shared-env-pilot"
)
CDR_VERSION = "v9"

# Confirmed via `ps -eo pid,args | grep gcsfuse` on the real VM: the mount for
# ~/workspace/cdrv9/vwb-aou-datasets-controlled-v9 is
# `gcsfuse --billing-project wb-swift-sprout-7231 ... vwb-aou-datasets-controlled ...`
# -- bucket name has no version suffix; v9's own subpath comes after it, matching
# the local mount's own subpath structure. The `--billing-project` flag on that
# gcsfuse invocation is a second signal worth acting on: this is a Requester Pays
# bucket, so any read (including from a Batch worker) needs a billing project
# specified or it fails regardless of IAM permissions -- see Stage 0/1/2 below,
# which all pass --billing-project explicitly rather than relying on dsub's own
# --input staging (unconfirmed whether it supports Requester Pays buckets at all).
ACAF_BUCKET_GS = "gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/acaf_threshold/pgen"

# Which base ancestry group to QC/thin -- must match 01_premade_label_filter.ipynb's
# own BASE_GROUP values ("eur", "afr").
BASE_GROUP = "eur"   # <-- change this and rerun for "eur" and "afr"

ANCESTRY_BUCKET_DIR = f"{WORKSPACE_BUCKET}/{CDR_VERSION}/01_ancestry_filtering"
ANCESTRY_BUCKET_DIR_GS = f"{WORKSPACE_BUCKET_GS}/{CDR_VERSION}/01_ancestry_filtering"

PREMADE_KEEP_PATH = f"{ANCESTRY_BUCKET_DIR}/premade_label_{CDR_VERSION}/premade_keep_ids_{BASE_GROUP}.txt"
PREMADE_KEEP_PATH_GS = f"{ANCESTRY_BUCKET_DIR_GS}/premade_label_{CDR_VERSION}/premade_keep_ids_{BASE_GROUP}.txt"
assert os.path.isfile(PREMADE_KEEP_PATH), f"premade-label keep-list not found: {PREMADE_KEEP_PATH!r} -- run 01_premade_label_filter.ipynb first"

# same output location 03a_genome_wide_qc_thinning_merge.ipynb writes to, so its
# own "Merge chromosomes" section works unchanged on whichever chromosomes were
# produced here instead
BUCKET_DIR_GS = f"{ANCESTRY_BUCKET_DIR_GS}/genome_wide_panel_{BASE_GROUP}"
PLINK_BIN_GS = f"{BUCKET_DIR_GS}/bin/plink2"

THIN_P = 0.2   # fixed from 00_optional_chr22_qc_thinning_timing.ipynb's calibration, same as 03a's

# per-task machine -- much lighter than GRM shard construction (single chromosome,
# not the full merged panel); adjust after Stage 1's real numbers, same convention
# as 03b_grm_shard_batch_submit.ipynb
MACHINE_VCPUS = 8
MEMORY_MB = 32768

print(ACAF_BUCKET_GS)
print(BUCKET_DIR_GS)

## Stage the plink2 binary (one-time)

Same lesson as `03b_grm_shard_batch_submit.ipynb`: the default Batch image has
neither `wget` nor `curl`, so stage the binary into the bucket once and let `dsub`
localize it via `--input`.

In [ ]:
%%bash
set -e

BIN_DIR="$HOME/bin"
mkdir -p "$BIN_DIR"

if [ ! -x "$BIN_DIR/plink2" ]; then
  PLINK2_URL="https://s3.amazonaws.com/plink2-assets/alpha7/plink2_linux_x86_64_20260504.zip"
  cd /tmp
  wget -q -O plink2.zip "$PLINK2_URL"
  unzip -o -q plink2.zip plink2 -d "$BIN_DIR"
  chmod +x "$BIN_DIR/plink2"
fi

"$BIN_DIR/plink2" --version

In [ ]:
%%bash -s "$PLINK_BIN_GS"
set -e
PLINK_BIN_GS=$1

local_plink="$HOME/bin/plink2"
if [ ! -x "$local_plink" ]; then
  echo "no local plink2 at $local_plink -- run the cell above first" >&2
  exit 1
fi

gcloud storage cp "$local_plink" "$PLINK_BIN_GS"
gcloud storage ls -l "$PLINK_BIN_GS"

## Stage 0 -- confirm Batch can actually read the CDR's ACAF bucket

**The real unknown this notebook exists to test -- now two-fold.** Not a plink job at all -- just `gcloud storage stat --billing-project` on one real ACAF chromosome object, run as the Batch service account, from a Batch worker. Two ways this can fail, both real:

1. **VPC-SC / IAM**: Batch workers might not be permitted to read controlled-tier CDR data directly at all (only the interactive VM's own gcsfuse mount might have that access, via a binding specific to it).
2. **Requester Pays**: the local gcsfuse mount for this bucket uses `--billing-project`, confirming `vwb-aou-datasets-controlled` is a Requester Pays bucket -- reads need a billing project specified or they fail regardless of IAM permissions. `--billing-project` is passed explicitly below rather than assumed to be handled automatically.

If this fails, check `status-detail` for which of the two it is before assuming the whole approach is a dead end -- a Requester Pays failure has a different fix (pass `--billing-project` everywhere, including inside Stage 1/2's script) than a VPC-SC failure (no fix, this approach can't work).

In [ ]:
%%bash -s "$PROJECT_ID" "$REGION" "$WORKSPACE_BUCKET_GS" "$CDR_VERSION" "$SERVICE_ACCOUNT" "$NETWORK" "$SUBNETWORK" "$ACAF_BUCKET_GS"
set -e
PROJECT_ID=$1
REGION=$2
WORKSPACE_BUCKET_GS=$3
CDR_VERSION=$4
SERVICE_ACCOUNT=$5
NETWORK=$6
SUBNETWORK=$7
ACAF_BUCKET_GS=$8

LOGGING_GS="${WORKSPACE_BUCKET_GS}/${CDR_VERSION}/01_ancestry_filtering/dsub_logs"

dsub \
  --provider google-batch \
  --project "$PROJECT_ID" \
  --regions "$REGION" \
  --logging "$LOGGING_GS" \
  --service-account "$SERVICE_ACCOUNT" \
  --network "$NETWORK" \
  --subnetwork "$SUBNETWORK" \
  --use-private-address \
  --name "acaf-access-check" \
  --env PROJECT_ID="$PROJECT_ID" \
  --env ACAF_TEST_PATH="${ACAF_BUCKET_GS}/acaf_threshold.chr22.pvar" \
  --command '
    echo "Checking read access to: $ACAF_TEST_PATH (billing project: $PROJECT_ID)"
    if gcloud storage stat --billing-project="$PROJECT_ID" "$ACAF_TEST_PATH"; then
      echo "SUCCESS: Batch worker can read the ACAF bucket (with --billing-project)"
    else
      echo "FAILED: Batch worker cannot read the ACAF bucket even with --billing-project set -- check status-detail: VPC-SC/IAM (no fix) vs. something else (e.g. wrong billing project)" >&2
      exit 1
    fi
  ' \
  > /tmp/acaf_check_job_id.txt

cat /tmp/acaf_check_job_id.txt

In [ ]:
%%bash -s "$PROJECT_ID" "$REGION"
set -e
PROJECT_ID=$1
REGION=$2
JOB_ID=$(tail -1 /tmp/acaf_check_job_id.txt)

dstat --provider google-batch --project "$PROJECT_ID" --location "$REGION" --jobs "$JOB_ID" --users 'jupyter' --status '*' --full

## Stage 1 -- single-chromosome validation

Only run once Stage 0 comes back `SUCCESS`. Runs the real QC+thin `plink2` command against one chromosome (chr22, the same one `00_optional_chr22_qc_thinning_timing.ipynb` calibrated `THIN_P` against). ACAF's 3 files are passed in as plain `gs://` paths via `--env` and downloaded manually inside the script with `gcloud storage cp --billing-project` (not dsub's own `--input`, since it's unconfirmed whether dsub's automatic localization supports Requester Pays buckets at all -- doing it explicitly here means the exact same command Stage 0 already proved works). Writes output to `BUCKET_DIR_GS` in the exact filename `03a_genome_wide_qc_thinning_merge.ipynb`'s merge section expects (`chr{N}_thinned_{CDR_VERSION}_{BASE_GROUP}.{pgen,pvar,psam}`).

In [ ]:
%%bash -s "$PROJECT_ID" "$REGION" "$WORKSPACE_BUCKET_GS" "$CDR_VERSION" "$SERVICE_ACCOUNT" "$NETWORK" "$SUBNETWORK" "$ACAF_BUCKET_GS" "$PREMADE_KEEP_PATH_GS" "$PLINK_BIN_GS" "$BUCKET_DIR_GS" "$THIN_P" "$MACHINE_VCPUS" "$MEMORY_MB" "$BASE_GROUP"
set -e
PROJECT_ID=$1
REGION=$2
WORKSPACE_BUCKET_GS=$3
CDR_VERSION=$4
SERVICE_ACCOUNT=$5
NETWORK=$6
SUBNETWORK=$7
ACAF_BUCKET_GS=$8
PREMADE_KEEP_PATH_GS=$9
PLINK_BIN_GS=${10}
BUCKET_DIR_GS=${11}
THIN_P=${12}
MACHINE_VCPUS=${13}
MEMORY_MB=${14}
BASE_GROUP=${15}

LOGGING_GS="${WORKSPACE_BUCKET_GS}/${CDR_VERSION}/01_ancestry_filtering/dsub_logs"
CHR=22
OUT_NAME="chr${CHR}_thinned_${CDR_VERSION}_${BASE_GROUP}"

dsub \
  --provider google-batch \
  --project "$PROJECT_ID" \
  --regions "$REGION" \
  --logging "$LOGGING_GS" \
  --service-account "$SERVICE_ACCOUNT" \
  --network "$NETWORK" \
  --subnetwork "$SUBNETWORK" \
  --use-private-address \
  --name "qc-thin-chr22-validation" \
  --machine-type "n1-standard-${MACHINE_VCPUS}" \
  --disk-size 100 \
  --input KEEP_PATH="$PREMADE_KEEP_PATH_GS" \
  --input PLINK_BIN="$PLINK_BIN_GS" \
  --env PROJECT_ID="$PROJECT_ID" \
  --env CHR_PGEN_GS="${ACAF_BUCKET_GS}/acaf_threshold.chr${CHR}.pgen" \
  --env CHR_PVAR_GS="${ACAF_BUCKET_GS}/acaf_threshold.chr${CHR}.pvar" \
  --env CHR_PSAM_GS="${ACAF_BUCKET_GS}/acaf_threshold.chr${CHR}.psam" \
  --env THIN_P="$THIN_P" \
  --env OUT_NAME="$OUT_NAME" \
  --output-recursive OUT_DIR="$BUCKET_DIR_GS" \
  --command '
    set -e
    chmod +x "$PLINK_BIN"

    # Requester Pays bucket -- --billing-project required, same command Stage 0
    # already proved works from a Batch worker
    gcloud storage cp --billing-project="$PROJECT_ID" "$CHR_PGEN_GS" chr.pgen
    gcloud storage cp --billing-project="$PROJECT_ID" "$CHR_PVAR_GS" chr.pvar
    gcloud storage cp --billing-project="$PROJECT_ID" "$CHR_PSAM_GS" chr.psam

    "$PLINK_BIN" \
      --pgen chr.pgen \
      --pvar chr.pvar \
      --psam chr.psam \
      --keep "$KEEP_PATH" \
      --thin '"$THIN_P"' \
      --set-all-var-ids "@:#:\$r:\$a" \
      --new-id-max-allele-len 1000 \
      --maf 0.01 \
      --hwe 1e-6 0.001 keep-fewhet \
      --geno 0.05 \
      --max-alleles 2 \
      --rm-dup exclude-all \
      --nonfounders \
      --threads '"$MACHINE_VCPUS"' \
      --memory '"$MEMORY_MB"' \
      --make-pgen \
      --out "${OUT_DIR}/${OUT_NAME}"
  ' \
  > /tmp/qc_validation_job_id.txt

cat /tmp/qc_validation_job_id.txt

In [ ]:
%%bash -s "$PROJECT_ID" "$REGION" "$BUCKET_DIR_GS"
set -e
PROJECT_ID=$1
REGION=$2
BUCKET_DIR_GS=$3
JOB_ID=$(tail -1 /tmp/qc_validation_job_id.txt)

dstat --provider google-batch --project "$PROJECT_ID" --location "$REGION" --jobs "$JOB_ID" --users 'jupyter' --status '*' --full

echo
echo "--- output files ---"
gcloud storage ls -l "${BUCKET_DIR_GS}/" 2>/dev/null | grep chr22 || echo "(none yet -- check task status above)"

## Stage 2 -- full run (all 22 chromosomes)

Only run once Stage 1 comes back `SUCCESS` with a real, sane-looking chr22 output
(compare its variant count against `03a_genome_wide_qc_thinning_merge.ipynb`'s own
interactive chr22 run, or `00_optional_chr22_qc_thinning_timing.ipynb`'s
calibration, if either already ran). One task per chromosome via a `--tasks` TSV
-- each task gets its own dedicated VM and its own network path to ACAF, so all
22 genuinely run in parallel rather than contending, unlike the interactive
driver.

In [ ]:
CHR_LENGTHS = {
    1: 248_956_422, 2: 242_193_529, 3: 198_295_559, 4: 190_214_555,
    5: 181_538_259, 6: 170_805_979, 7: 159_345_973, 8: 145_138_636,
    9: 138_394_717, 10: 133_797_422, 11: 135_086_622, 12: 133_275_309,
    13: 114_364_328, 14: 107_043_718, 15: 101_991_189, 16: 90_338_345,
    17: 83_257_441, 18: 80_373_285, 19: 58_617_616, 20: 64_444_167,
    21: 46_709_983, 22: 50_818_468,
}
CHRS_LARGEST_FIRST = sorted(CHR_LENGTHS, key=CHR_LENGTHS.get, reverse=True)

# gs:// paths passed as plain --env strings, not --input -- Stage 1 confirmed
# manual `gcloud storage cp --billing-project` inside the script is what actually
# works against this Requester Pays bucket, so Stage 2 uses the same mechanism
TASKS_PATH = "/tmp/qc_thin_tasks.tsv"
with open(TASKS_PATH, "w") as f:
    f.write("--env CHR\t--env CHR_PGEN_GS\t--env CHR_PVAR_GS\t--env CHR_PSAM_GS\t--env OUT_NAME\n")
    for chr_num in CHRS_LARGEST_FIRST:
        out_name = f"chr{chr_num}_thinned_{CDR_VERSION}_{BASE_GROUP}"
        f.write(
            f"{chr_num}\t{ACAF_BUCKET_GS}/acaf_threshold.chr{chr_num}.pgen\t"
            f"{ACAF_BUCKET_GS}/acaf_threshold.chr{chr_num}.pvar\t"
            f"{ACAF_BUCKET_GS}/acaf_threshold.chr{chr_num}.psam\t{out_name}\n"
        )

print(open(TASKS_PATH).read())

In [ ]:
%%bash -s "$PROJECT_ID" "$REGION" "$WORKSPACE_BUCKET_GS" "$CDR_VERSION" "$SERVICE_ACCOUNT" "$NETWORK" "$SUBNETWORK" "$PREMADE_KEEP_PATH_GS" "$PLINK_BIN_GS" "$BUCKET_DIR_GS" "$THIN_P" "$MACHINE_VCPUS" "$MEMORY_MB"
set -e
PROJECT_ID=$1
REGION=$2
WORKSPACE_BUCKET_GS=$3
CDR_VERSION=$4
SERVICE_ACCOUNT=$5
NETWORK=$6
SUBNETWORK=$7
PREMADE_KEEP_PATH_GS=$8
PLINK_BIN_GS=$9
BUCKET_DIR_GS=${10}
THIN_P=${11}
MACHINE_VCPUS=${12}
MEMORY_MB=${13}

LOGGING_GS="${WORKSPACE_BUCKET_GS}/${CDR_VERSION}/01_ancestry_filtering/dsub_logs"

dsub \
  --provider google-batch \
  --project "$PROJECT_ID" \
  --regions "$REGION" \
  --logging "$LOGGING_GS" \
  --service-account "$SERVICE_ACCOUNT" \
  --network "$NETWORK" \
  --subnetwork "$SUBNETWORK" \
  --use-private-address \
  --name "qc-thin-genome-wide" \
  --machine-type "n1-standard-${MACHINE_VCPUS}" \
  --disk-size 100 \
  --input KEEP_PATH="$PREMADE_KEEP_PATH_GS" \
  --input PLINK_BIN="$PLINK_BIN_GS" \
  --env PROJECT_ID="$PROJECT_ID" \
  --output-recursive OUT_DIR="$BUCKET_DIR_GS" \
  --command '
    set -e
    chmod +x "$PLINK_BIN"

    # Requester Pays bucket -- --billing-project required, same mechanism
    # Stage 0/1 already proved works from a Batch worker
    gcloud storage cp --billing-project="$PROJECT_ID" "$CHR_PGEN_GS" chr.pgen
    gcloud storage cp --billing-project="$PROJECT_ID" "$CHR_PVAR_GS" chr.pvar
    gcloud storage cp --billing-project="$PROJECT_ID" "$CHR_PSAM_GS" chr.psam

    "$PLINK_BIN" \
      --pgen chr.pgen \
      --pvar chr.pvar \
      --psam chr.psam \
      --keep "$KEEP_PATH" \
      --thin '"$THIN_P"' \
      --set-all-var-ids "@:#:\$r:\$a" \
      --new-id-max-allele-len 1000 \
      --maf 0.01 \
      --hwe 1e-6 0.001 keep-fewhet \
      --geno 0.05 \
      --max-alleles 2 \
      --rm-dup exclude-all \
      --nonfounders \
      --threads '"$MACHINE_VCPUS"' \
      --memory '"$MEMORY_MB"' \
      --make-pgen \
      --out "${OUT_DIR}/${OUT_NAME}"
  ' \
  --tasks /tmp/qc_thin_tasks.tsv \
  > /tmp/qc_genome_wide_job_id.txt

cat /tmp/qc_genome_wide_job_id.txt

In [ ]:
%%bash -s "$PROJECT_ID" "$REGION"
set -e
PROJECT_ID=$1
REGION=$2
JOB_ID=$(cat /tmp/qc_genome_wide_job_id.txt)

dstat --provider google-batch --project "$PROJECT_ID" --location "$REGION" --jobs "$JOB_ID" --users 'jupyter' --status '*' --full

## Next steps

Once all 22 tasks come back `SUCCESS` and `${BUCKET_DIR_GS}/` has all 22
`chr{N}_thinned_{CDR_VERSION}_{BASE_GROUP}.{pgen,pvar,psam}` trios, run
`03a_genome_wide_qc_thinning_merge.ipynb`'s **"Merge chromosomes into a genome-wide
pfile"** section directly -- it only checks `BUCKET_DIR` for already-persisted
chromosome trios and doesn't care whether they came from that notebook's own
driver or from here, so no changes needed there. Skip that notebook's own driver
cell entirely once this one has produced all 22.